In [16]:
# transferable code functions
import numpy as np
import scipy.signal as scisig


def padding(layer:np.ndarray, mode:str = 'zero', pad_size:int = 1):

    for i in range(pad_size):

        padded_ly_size = (layer.shape[0]+2, layer.shape[1]+2)
        padded_ly = np.zeros(padded_ly_size)
        padded_ly[1:-1,1:-1] = layer
    
        if mode == 'continue':
            padded_ly[0,1:-1] = layer[0,:]
            padded_ly[-1,1:-1] = layer[-1,:]

            padded_ly[1:-1,0] = layer[:,0]
            padded_ly[1:-1,-1] = layer[:,-1]

            padded_ly[0,0] = layer[0,0]
            padded_ly[0,-1] = layer[0,-1]
            padded_ly[-1,0] = layer[-1,0]
            padded_ly[-1,-1] = layer[-1,-1]
        layer = padded_ly

    return(padded_ly)

def convolve(input:np.ndarray, kernel:np.ndarray, pad_mode:str = 'zero', pad_size:int = 666, conv_mode:str="valid"):
    if pad_mode !="none":
        if pad_size == 666:
            pad_size = kernel.shape[0]//2
        input = padding(input, pad_mode, pad_size)
    k_shp = kernel.shape
    #print(input.shape)
    result = scisig.convolve(input, kernel, mode=conv_mode, method="fft")

    return(result)

def pool(input:np.ndarray, stride = 2, mode = "max"):
    shape = (int(input.shape[0]/stride), int(input.shape[1]/stride))
    output = np.zeros(shape)
    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            if mode =="max":
                output[i,j] = np.max(input[i*stride:i*stride+stride,j*stride:j*stride+stride])
                #look at that absolute index fuckery
            if mode =="mean":
                output[i,j] = np.mean(input[i*stride:i*stride+stride,j*stride:j*stride+stride])
    return(output)

def sigmond(input):
    return( 1/(1+np.exp(-input)))

def sigmond_prime(input):
    return(sigmond(input) * (1-sigmond(input)))

def ReLU(input):
    return(np.maximum(0, input))

def ReLU_prime(input):
    return(input > 0).astype(input.dtype)



In [17]:
tup = (4,8)
out = (tup[0]/2,tup[1]/2)
print(out)

inp = np.array([[1,2],[3,4]])
outp= pool(inp, 2, "max")
print(outp)

(2.0, 4.0)
[[4.]]


In [18]:
test_arr  = np.zeros((4))
test_arr[:] = [-3, -1, 0.00, 1.34]

result1 = ReLU(test_arr)
result2 = ReLU_prime(test_arr)
print(result1, result2)

[0.   0.   0.   1.34] [0. 0. 0. 1.]


In [19]:
input =  np.ones((3,3)) 
input_even =  np.ones((4,4))

kernel = np.zeros(shape=(3,3))
kernel[:,:] = [[0.125, 0.25, 0.125], #making a simple filter so we can see it's effects
                  [0.250, 1.00, 0.250],
                  [0.125, 0.25, 0.125]]

kernel_even = np.zeros(shape=(2,2))
kernel_even[:,:] = [[0.125, 0.25], #making a simple filter so we can see it's effects
                  [0.250, 1.00]]


convolved = convolve(input, kernel)
print(convolved)

k_shp = kernel.shape
k_shp = (5,)
padding_size = k_shp[0]//2
print(padding_size)



[[1.625 2.    1.625]
 [2.    2.5   2.   ]
 [1.625 2.    1.625]]
2


In [20]:
import hydra
from omegaconf import DictConfig, OmegaConf
cfg = OmegaConf.load("model.yaml")


In [21]:
#todo I'd like to create a version of this that can have dynamically sized convolutional layers
import numpy as np
from dataclasses import dataclass
from omegaconf import DictConfig
import scipy.signal as scisig


def build_index_lookup(cfg: DictConfig):
    """
    Given a loaded OmegaConf config with a 'layers' section,
    build a lookup table: index -> (layer_name, layer_data).
    """
    blocks = cfg.blocks

    index_lookup = {
        block_data.index: (block_name, block_data)
        for block_name, block_data in blocks.items()
    }
    return index_lookup

class Layer:
    #need to refactor this into a block that includes weights
    def __init__(self, ltype:str, index:int,  activations: np.ndarray, z_values: np.ndarray, dims:int, shape: tuple):
        self.activations = activations
        self.z_values = z_values
        self.ltype = ltype
        self.shape = shape
        self.dims = dims
        self.index = index

    @classmethod
    def from_shape(cls, layer_shape, l_type, index, dtype=np.float32, **kwargs):
        activations = np.zeros(layer_shape, dtype=dtype)
        z_vals = np.zeros(layer_shape, dtype=dtype)
        shape = activations.shape
        ltype = l_type
        dims = len(shape)
        index = index
        return cls(ltype, index, activations, z_vals, dims, shape, **kwargs)

class Input_block:
    def __init__(self, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index

        self.b_type = "input"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)

class FC_block:
    def __init__(self, prev_ly_shape, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index
        self.prev_ly_shape = prev_ly_shape

        self.b_type = "fc"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)
        self.biases = np.zeros(layer_shape)
        self.weights = self._init_weights()
        self.fc_weights_shape = self.weights.shape

    def _init_weights(self, init=True):
        if init:
            weights = np.random.uniform(-1,1, size=(*self.prev_ly_shape, *self.layer_shape))
        else:
            weights = np.zeros(shape=(*self.prev_ly_shape, *self.layer_shape))
        return(weights)
    
    def forward(self, input):
        self.z_values = np.tensordot(input, self.weights, axes=((0,1),(0,1)))  + self.biases
        self.activations = ReLU(self.z_values)

class FC_CONV_block:
    def __init__(self, prev_bk_shape, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index
        self.prev_bk_shape = prev_bk_shape

        self.b_type = "fc_conv"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)
        self.biases = np.zeros(layer_shape)
        self.weights = self._init_weights()
        self.fc_weights_shape = self.weights.shape

    def _init_weights(self, init=True):
        if init:
            weights = np.random.uniform(-1,1, size=(*self.prev_bk_shape, *self.layer_shape))
        else:
            weights = np.zeros(shape=(*self.prev_bk_shape, *self.layer_shape))
        return(weights)
    
    def forward(self, input):

        #temp_z_lys = np.zeros(shape=(input.shape[0], *self.layer_shape))
        #for i in range(input.shape[0]):
            #for each input feature map:
        #    temp_z_lys[i,:,:] = np.dot(input[i, :,:], self.weights[i,:,:,:,:]) #not finished!
        #self.z_values = np.sum(temp_z_lys, axis=0) + self.biases
        self.z_values = np.tensordot(input, self.weights, axes=((0,1,2),(0,1,2)))
        self.activations = ReLU(self.z_values)

class Conv_Block:
    def __init__(self, num_filters, kernel_shape, stride, layer_shape, index):
        self.num_filters = num_filters
        self.kernel_shape = kernel_shape
        self.stride = stride
        self.layer_shape = layer_shape
        self.index = index

        self.b_type = "conv"
        self.ly_dim = len(layer_shape)
        self.filters = self.init_filters(self.num_filters, self.kernel_shape, init = True)
        self.feature_maps, self.feature_map_z_vals = self._init_feature_maps()
        self.feature_map_biases = np.zeros_like(self.feature_maps)
        self.biases = np.zeros(layer_shape) #idk if i need this but we can remove it later
        self.activations = self.feature_maps
        self.z_vals = self.feature_map_z_vals

    def _init_feature_maps(self):
        feature_maps = np.zeros(shape=(self.num_filters, *self.layer_shape))
        feature_map_z_vals = np.zeros_like(feature_maps)  
        return(feature_maps, feature_map_z_vals)

    @staticmethod    
    def init_filters(num_filters:int, kernel_shape:tuple, init=True):
        if init:
            filters = np.random.uniform(-1,1, size=(num_filters, *kernel_shape))
        else:
            filters = np.zeros((num_filters, *kernel_shape))
        return (filters)
    
    def forward(self, input:np.ndarray):
        #input should be an ndarray of num_channels X hieght X width
        for i in range(self.feature_maps.shape[0]):
            if len(input.shape) == 2:
                input = np.expand_dims(input, axis=0)
            temp_maps = np.zeros_like(input)
            for j in range(input.shape[0]):
                temp_maps[j, :, :] = convolve(input[j,:,:], self.filters[i,:,:], conv_mode="valid")
            self.feature_map_z_vals = np.sum(temp_maps, axis=0) + self.feature_map_biases[i, :,:]
            self.feature_maps[i, :,:] = ReLU(self.feature_map_z_vals)
            self.activations = self.feature_maps
            self.z_vals = self.feature_map_z_vals

class Pooling_ly:
    def __init__(self, shape, index, stride:int=2, p_type:str="max"):
        self.index = index
        self.shape = shape
        self.stride = stride
        self.p_type = p_type
        self.b_type = "pooling"


        self.activations = np.ndarray(shape=(2,2,2))
        self.z_vals = np.zeros_like(self.activations)

    def pooling(self, input_act):
        if len(input_act.shape) == 2:
            input_act = np.expand_dims(input_act, axis=0)
        
        print(self.shape)
        self.activations = np.zeros((input_act.shape[0], *self.shape))
        for i in range(input_act.shape[0]):
            self.activations[i,:,:] = pool(input_act[i,:,:], stride=self.stride, mode=self.p_type)
        

    def forward(self, input):
        self.pooling(input)
        self.z_vals = self.activations

 

class NN:
    def __init__(self, config:DictConfig, blocks: list):
        self.config = config
        self.blocks = blocks

    @classmethod
    def create_network(cls, cfg:DictConfig, **kwargs):
        index_to_block = build_index_lookup(cfg)
        print("creating network....")

        blocks = []
        for items in cfg.blocks:
            print(items)
            block_name, block_data = index_to_block[cfg.blocks[items].index]
            

            if block_data.type == "input":
                blocks.append(Input_block(block_data.shape, block_data.index))
            if block_data.type == "fc":
                if prev_bk_data.type == "conv2D":
                    prev_bk_shp = (prev_bk_data.filters.filter_num, *prev_bk_data.shape)
                    print(prev_bk_shp)
                    blocks.append(FC_CONV_block(prev_bk_shp, block_data.shape, block_data.index))
                else:
                    blocks.append(FC_block(prev_bk_data.shape, block_data.shape, block_data.index))
            if block_data.type == "pool":
                blocks.append(Pooling_ly(block_data.shape, block_data.index, block_data.stride, block_data.mode))

            if block_data.type == "conv2D":
                fltr = block_data.filters
                blocks.append(Conv_Block(fltr.filter_num, fltr.kernel_shape, fltr.stride, block_data.shape, block_data.index))

            prev_bk_name = block_name
            prev_bk_data = block_data

        print("Done!")
        return cls(cfg, blocks)
    
    def forward(self, input_ly):
        print("running forward function....")
        self.blocks[0].activations = input_ly
        for index in range(len(self.blocks)):
            print("\nindex: ", index)
            if index == 0:
                self.blocks[0].activations = input_ly
            else:
                self.blocks[index].forward(self.blocks[index-1].activations)
        print("Done!")
            




#todo:
    #figure out network autocreation DONE!
    #figure out padding algorrithm DONE!
    #figure out down sizing 
        #this will be through pooling
    #write forward functions DONE!
        #since these differ between different types of layers and blocks, 
        # maybe each block should have a forward function?



In [22]:
cfg = OmegaConf.load("model.yaml")

model = NN.create_network(cfg)

input_ly = np.random.rand(28,28)
model.forward(input_ly)

creating network....
input_ly
conv_block_1
pool_block_1
conv_block_2
conv_block_3
fc_ly_1
(27, 14, 14)
output_ly
Done!
running forward function....

index:  0

index:  1

index:  2
[14, 14]

index:  3

index:  4

index:  5

index:  6
Done!


In [23]:
print(model.blocks[2].activations)

[[[0.         0.         0.         ... 0.         0.         0.33313157]
  [0.         0.         0.         ... 0.         0.         0.        ]
  [0.         0.         0.         ... 0.         0.         0.66092537]
  ...
  [0.         0.19656424 0.         ... 0.         0.         0.        ]
  [0.         0.         0.00615707 ... 0.         0.         0.        ]
  [0.         0.17232667 0.         ... 0.08761886 0.         0.46756158]]

 [[1.00651386 1.6754117  1.05860685 ... 1.31251422 0.80056268 0.56443571]
  [1.21300054 1.20954758 1.3270818  ... 1.99137951 0.54754498 0.83388678]
  [1.06055495 1.01705489 1.03952025 ... 2.02243639 1.33876888 0.76606228]
  ...
  [0.99196474 1.10419316 1.43049712 ... 1.09545985 1.16693476 1.1090744 ]
  [1.38092593 1.5058779  0.91903675 ... 1.48533948 1.5658684  1.95263447]
  [1.30415695 1.19466501 1.09725672 ... 1.14576061 0.91263398 1.10853989]]

 [[0.         0.         0.         ... 0.         0.         0.        ]
  [0.25875545 0.      

In [24]:
item = model.blocks
print("blocks")
for items in item:
    print(items.activations.shape, items.index, items.b_type)




blocks
(28, 28) 0 input
(27, 28, 28) 1 conv
(27, 14, 14) 2 pooling
(27, 14, 14) 3 conv
(27, 14, 14) 4 conv
(10, 10) 5 fc_conv
(10,) 6 fc


In [ ]:
cfg = OmegaConf.load("model.yaml")
print(type(cfg))
print(cfg)
examine_1 = cfg.model.layers.input_ly.shape
print("\nexamine_1: ")
print(examine_1)
print(type(examine_1))
examine_2 = tuple(cfg.model.layers.input_ly.shape)
print("\nexamine_2: ")
print(examine_2)
print(type(examine_2))
#next to figure out how to handle .yaml files and DictConfig files

#net = NN.create_network(cfg)


In [ ]:
import numpy as np
filters  = {"prev_ly_size":28,
            "kernel_size": 3,
            "stride" : 1}
l1 = np.zeros(shape=(filters["prev_ly_size"],))
l11 = np.zeros(shape=(filters["prev_ly_size"],))

l2 = []
l1[0] = 1
for index, values in enumerate(l1):
    if index%filters["stride"] == 0:
        l1[index] = 1


    if l1[index] == 1:
        for x in range(filters["kernel_size"]): 
            y=x+1
            try:
                l11[index + (y - filters["kernel_size"]//2)] = l11[index + (y - filters["kernel_size"]//2)] + 1
            except IndexError as error:
                print("had an index error, continuing")
print(l1,"\n",l11)